# 04 · Catálogo de datos – Olist
**Alcance:** catálogo `olist`, esquemas `bronze`, `silver` y `gold`

Este notebook completa la **documentación en Unity Catalog**:
1. Descripción y etiqueta `capa` en cada tabla de **Bronze** (Silver y Gold ya las ponen al crearse).
2. Descripción de **cada columna** de las tres capas, incluidas las tablas de cuarentena, los logs y la vista pública.
3. El **diccionario de datos** (esquema, tabla, columna, tipo y descripción) consultado desde `information_schema`, más un control de cobertura.

> **Va al final del Workflow** (`bronze → silver → gold → catalogo`): los otros notebooks reescriben sus tablas en cada ejecución, y eso puede borrar las descripciones de columnas. Así se vuelven a poner cada vez.

## 1. Parámetros

In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalogo", "olist")
catalogo = dbutils.widgets.get("catalogo")
CAPAS = ["bronze", "silver", "gold"]


def sql_texto(texto):
    """Escapa comillas simples para usar el texto dentro de un literal SQL."""
    return texto.replace("'", "\\'")


print("Catálogo:", catalogo)

## 2. Descripción de las tablas Bronze
Bronze guarda el dato tal como llega, pero eso no exime de documentarlo: quien busque en el catálogo debe saber qué contiene cada tabla cruda.

In [0]:
TABLAS_BRONZE = {
    "orders":               "Pedidos crudos de Olist: estado y fechas del ciclo de compra. Todo como texto",
    "order_items":          "Items crudos de cada pedido: producto, vendedor, precio y flete. Todo como texto",
    "order_payments":       "Pagos crudos por pedido: medio, cuotas y monto. Todo como texto",
    "order_reviews":        "Reseñas crudas de clientes: puntaje y comentarios. Todo como texto",
    "customers":            "Clientes crudos: id por pedido, id de persona y ubicación. Todo como texto",
    "sellers":              "Vendedores crudos del marketplace y su ubicación. Todo como texto",
    "products":             "Productos crudos: categoría en portugués y medidas. Todo como texto",
    "geolocation":          "Coordenadas crudas por prefijo postal, varias por prefijo. Todo como texto",
    "category_translation": "Traducción cruda de categorías, portugués a inglés",
    "log_ingesta":          "Historial de ejecuciones de la ingesta: filas cargadas por tabla",
}

for tabla, descripcion in TABLAS_BRONZE.items():
    nombre = f"{catalogo}.bronze.{tabla}"
    spark.sql(f"COMMENT ON TABLE {nombre} IS '{sql_texto(descripcion)}'")
    spark.sql(f"ALTER TABLE {nombre} SET TAGS ('capa' = 'bronze', 'dominio' = 'ecommerce')")

print(f"{len(TABLAS_BRONZE)} tablas Bronze documentadas y etiquetadas")

## 3. Descripciones de columnas
Dos diccionarios, escritos **una sola vez**:
- `COLUMNAS_BRONZE`: nombres originales de Kaggle (en inglés, todo texto).
- `COLUMNAS_NEGOCIO`: nombres de negocio de Silver y Gold. Una misma columna (`pedido_id`, `estado`, …) significa lo mismo en todas las tablas donde aparece, así que se describe una vez.

`POR_TABLA` cubre los pocos casos en que el mismo nombre significa algo distinto según la tabla.

In [0]:
CRUDO = " (texto crudo de Kaggle)"

COLUMNAS_BRONZE = {
    "order_id":                      "Identificador del pedido" + CRUDO,
    "customer_id":                   "Id del cliente para un pedido; Olist genera uno por pedido" + CRUDO,
    "order_status":                  "Estado del pedido: delivered, shipped, canceled, etc." + CRUDO,
    "order_purchase_timestamp":      "Fecha y hora de la compra" + CRUDO,
    "order_approved_at":             "Fecha y hora de aprobación del pago" + CRUDO,
    "order_delivered_carrier_date":  "Fecha y hora de entrega a la transportista" + CRUDO,
    "order_delivered_customer_date": "Fecha y hora de entrega al cliente" + CRUDO,
    "order_estimated_delivery_date": "Fecha de entrega prometida al cliente" + CRUDO,
    "order_item_id":                 "Número correlativo del item dentro del pedido" + CRUDO,
    "product_id":                    "Identificador del producto" + CRUDO,
    "seller_id":                     "Identificador del vendedor" + CRUDO,
    "shipping_limit_date":           "Fecha límite de despacho del vendedor" + CRUDO,
    "price":                         "Precio del item en BRL" + CRUDO,
    "freight_value":                 "Costo de flete del item en BRL" + CRUDO,
    "payment_sequential":            "Orden del pago dentro del pedido" + CRUDO,
    "payment_type":                  "Medio de pago" + CRUDO,
    "payment_installments":          "Número de cuotas" + CRUDO,
    "payment_value":                 "Monto pagado en BRL" + CRUDO,
    "review_id":                     "Identificador de la reseña" + CRUDO,
    "review_score":                  "Puntaje de satisfacción de 1 a 5" + CRUDO,
    "review_comment_title":          "Título del comentario" + CRUDO,
    "review_comment_message":        "Texto del comentario" + CRUDO,
    "review_creation_date":          "Fecha de envío de la encuesta" + CRUDO,
    "review_answer_timestamp":       "Fecha y hora de respuesta de la encuesta" + CRUDO,
    "customer_unique_id":            "Id de la persona compradora" + CRUDO,
    "customer_zip_code_prefix":      "Prefijo postal del cliente" + CRUDO,
    "customer_city":                 "Ciudad del cliente" + CRUDO,
    "customer_state":                "Estado (UF) del cliente" + CRUDO,
    "seller_zip_code_prefix":        "Prefijo postal del vendedor" + CRUDO,
    "seller_city":                   "Ciudad del vendedor" + CRUDO,
    "seller_state":                  "Estado (UF) del vendedor" + CRUDO,
    "product_category_name":         "Categoría del producto en portugués" + CRUDO,
    "product_category_name_english": "Categoría del producto en inglés" + CRUDO,
    "product_name_lenght":           "Largo del nombre del producto; la errata viene del origen" + CRUDO,
    "product_description_lenght":    "Largo de la descripción; la errata viene del origen" + CRUDO,
    "product_photos_qty":            "Cantidad de fotos del producto" + CRUDO,
    "product_weight_g":              "Peso en gramos" + CRUDO,
    "product_length_cm":             "Largo en centímetros" + CRUDO,
    "product_height_cm":             "Alto en centímetros" + CRUDO,
    "product_width_cm":              "Ancho en centímetros" + CRUDO,
    "geolocation_zip_code_prefix":   "Prefijo postal" + CRUDO,
    "geolocation_lat":               "Latitud del punto" + CRUDO,
    "geolocation_lng":               "Longitud del punto" + CRUDO,
    "geolocation_city":              "Ciudad del punto" + CRUDO,
    "geolocation_state":             "Estado (UF) del punto" + CRUDO,
    "_ingestion_ts":                 "Auditoría: momento en que la fila entró a Bronze",
    "_source_file":                  "Auditoría: archivo del volume del que salió la fila",
    "tabla":                         "Tabla Bronze cargada",
    "filas":                         "Filas cargadas en esa ejecución",
    "fecha_ingesta":                 "Momento de la ejecución de la ingesta",
}

COLUMNAS_NEGOCIO = {
    # --- Claves e identificadores
    "pedido_id":              "Identificador del pedido",
    "cliente_id":             "Id del cliente para un pedido. Olist genera uno nuevo por cada pedido",
    "cliente_unico_id":       "Id de la persona compradora. Usar este para contar clientes reales",
    "producto_id":            "Identificador del producto",
    "vendedor_id":            "Identificador del vendedor",
    "resena_id":              "Identificador de la reseña. Se repite entre pedidos: la clave es resena_id + pedido_id",
    "item_num":               "Número correlativo del item dentro del pedido (1, 2, 3...)",
    "secuencia":              "Orden del pago dentro del pedido, cuando se paga en varias partes",
    # --- Ubicación
    "cp_prefijo":             "Primeros 5 dígitos del código postal (CEP)",
    "ciudad":                 "Ciudad, en minúsculas",
    "estado":                 "Estado de Brasil, sigla de 2 letras (UF)",
    "latitud":                "Latitud promedio del prefijo postal, en grados",
    "longitud":               "Longitud promedio del prefijo postal, en grados",
    "puntos_origen":          "Cantidad de puntos de Bronze consolidados en esta fila",
    # --- Producto
    "categoria_pt":           "Categoría del producto en portugués, como viene de Olist",
    "categoria_en":           "Categoría en inglés. Si no hay traducción se conserva el nombre en portugués",
    "categoria":              "Categoría del producto en inglés. sin_categoria si el producto no tiene una",
    "largo_nombre":           "Cantidad de caracteres del nombre del producto",
    "largo_descripcion":      "Cantidad de caracteres de la descripción del producto",
    "cant_fotos":             "Cantidad de fotos publicadas del producto",
    "peso_g":                 "Peso del producto en gramos",
    "largo_cm":               "Largo del producto en centímetros",
    "alto_cm":                "Alto del producto en centímetros",
    "ancho_cm":               "Ancho del producto en centímetros",
    # --- Pedido y entrega
    "estado_pedido":          "Estado del pedido en el flujo de Olist: delivered, shipped, canceled, etc.",
    "fecha_compra":           "Fecha y hora en que el cliente hizo la compra",
    "fecha_aprobacion":       "Fecha y hora de aprobación del pago",
    "fecha_envio":            "Fecha y hora en que el vendedor entregó el pedido a la transportista",
    "fecha_entrega":          "Fecha y hora de entrega al cliente. NULL si aún no se entregó",
    "fecha_entrega_estimada": "Fecha de entrega prometida al cliente al momento de la compra",
    "fecha_limite_envio":     "Fecha límite para que el vendedor despache el item",
    # --- Importes y pagos (BRL)
    "precio":                 "Precio del item en reales (BRL), sin flete",
    "flete":                  "Costo de envío del item en reales (BRL)",
    "tipo_pago":              "Medio de pago: credit_card, boleto, voucher o debit_card",
    "cuotas":                 "Número de cuotas del pago",
    "monto":                  "Monto pagado en reales (BRL)",
    # --- Reseñas
    "puntaje":                "Satisfacción del cliente, de 1 a 5",
    "titulo":                 "Título del comentario de la reseña. Opcional",
    "comentario":             "Texto del comentario de la reseña. Opcional",
    "fecha_creacion":         "Fecha en que se envió la encuesta de satisfacción al cliente",
    "fecha_respuesta":        "Fecha y hora en que el cliente respondió la encuesta",
    # --- Auditoría y calidad
    "_ingestion_ts":          "Auditoría: momento en que la fila entró a Bronze",
    "_procesado_ts":          "Auditoría: momento en que la fila se escribió en Silver",
    "_cuarentena_ts":         "Auditoría: momento en que la fila fue enviada a cuarentena",
    "motivos_rechazo":        "Lista de reglas de calidad que la fila no cumplió",
    "tabla":                  "Tabla Silver evaluada",
    "filas_bronze":           "Filas leídas de Bronze",
    "filas_silver":           "Filas escritas en Silver",
    "filas_cuarentena":       "Filas enviadas a cuarentena",
    "filas_consolidadas":     "Filas sanas eliminadas por duplicado o agrupadas (geolocalización)",
    "cuadra":                 "True si filas_bronze = filas sanas + filas_cuarentena",
    "fecha_proceso":          "Momento de la ejecución de Silver",
    # --- Gold: dimensiones
    "anio_mes":               "Mes de la compra, formato yyyy-MM",
    "cliente_estado":         "Estado (UF) del cliente. desconocido si no se pudo determinar",
    "cliente_ciudad":         "Ciudad del cliente. desconocido si no se pudo determinar",
    "vendedor_estado":        "Estado (UF) del vendedor. desconocido si no se pudo determinar",
    "tramo_retraso":          "Tramo de días de retraso respecto a la fecha prometida",
    # --- Gold: métricas
    "total_item":             "Ingreso del item: precio + flete, en BRL",
    "dias_entrega":           "Días entre la compra y la entrega al cliente",
    "dias_retraso":           "Días entre la fecha prometida y la entrega real. Negativo = llegó antes",
    "entregado_a_tiempo":     "True si se entregó en o antes de la fecha prometida (comparación por día)",
    "puntaje_resena":         "Puntaje medio de las reseñas del pedido, de 1 a 5",
    "pedidos":                "Cantidad de pedidos distintos",
    "clientes":               "Cantidad de clientes únicos (personas)",
    "items":                  "Cantidad de items vendidos",
    "ingresos":               "Suma de precio + flete, en BRL",
    "ticket_promedio":        "Ingreso promedio por pedido, en BRL",
    "participacion_pct":      "Porcentaje de los ingresos totales que aporta la categoría",
    "precio_promedio":        "Precio promedio por item, en BRL, sin flete",
    "puntaje_medio":          "Puntaje medio de las reseñas, de 1 a 5",
    "puesto":                 "Posición en el ranking por ingresos (1 = mayor)",
    "pedidos_entregados":     "Cantidad de pedidos con fecha de entrega",
    "dias_entrega_promedio":  "Promedio de días entre compra y entrega",
    "pct_a_tiempo":           "Porcentaje de pedidos entregados a tiempo",
    "pct_resenas_negativas":  "Porcentaje de pedidos con puntaje 1 o 2",
}

# Mismo nombre, distinto significado según la tabla: (esquema, tabla, columna) -> descripción
POR_TABLA = {
    ("silver", "clientes", "ciudad"):          "Ciudad del cliente, en minúsculas",
    ("silver", "clientes", "estado"):          "Estado (UF) del cliente, sigla de 2 letras",
    ("silver", "vendedores", "ciudad"):        "Ciudad del vendedor, en minúsculas",
    ("silver", "vendedores", "estado"):        "Estado (UF) del vendedor, sigla de 2 letras",
    ("silver", "geolocalizacion", "ciudad"):   "Ciudad más frecuente del prefijo postal",
    ("silver", "geolocalizacion", "estado"):   "Estado (UF) más frecuente del prefijo postal",
    ("silver", "geolocalizacion_cuarentena", "latitud"):  "Latitud del punto rechazado, en grados",
    ("silver", "geolocalizacion_cuarentena", "longitud"): "Longitud del punto rechazado, en grados",
    ("gold", "ventas_mensuales", "clientes"):  "Clientes únicos (personas) que compraron en el mes",
}

print(f"Descripciones: {len(COLUMNAS_BRONZE)} Bronze, {len(COLUMNAS_NEGOCIO)} negocio, {len(POR_TABLA)} por tabla")

## 4. Aplicar las descripciones
Se leen **todas** las columnas de las tres capas desde `information_schema.columns`, así no se escapa ninguna tabla (cuarentenas, logs, vista). Cada tabla se documenta en una sola sentencia `ALTER TABLE ... ALTER COLUMN ... COMMENT` por columna agrupada. Si una sentencia falla (por ejemplo, en la vista), se intenta columna por columna con `COMMENT ON COLUMN` y se registra el error sin detener el notebook.

In [0]:
columnas = spark.sql(f"""
    SELECT c.table_schema, c.table_name, c.column_name, t.table_type
    FROM {catalogo}.information_schema.columns c
    JOIN {catalogo}.information_schema.tables t
      ON t.table_schema = c.table_schema AND t.table_name = c.table_name
    WHERE c.table_schema IN ('bronze', 'silver', 'gold')
    ORDER BY c.table_schema, c.table_name, c.ordinal_position""").collect()


def descripcion(esquema, tabla, columna):
    if (esquema, tabla, columna) in POR_TABLA:
        return POR_TABLA[(esquema, tabla, columna)]
    base = COLUMNAS_BRONZE if esquema == "bronze" else COLUMNAS_NEGOCIO
    return base.get(columna)


por_tabla = {}
sin_descripcion = []
for r in columnas:
    texto = descripcion(r.table_schema, r.table_name, r.column_name)
    if texto is None:
        sin_descripcion.append(f"{r.table_schema}.{r.table_name}.{r.column_name}")
        continue
    clave = (r.table_schema, r.table_name, r.table_type)
    por_tabla.setdefault(clave, []).append((r.column_name, texto))

errores = []
documentadas = 0
for (esquema, tabla, tipo), cols in por_tabla.items():
    nombre = f"{catalogo}.{esquema}.{tabla}"
    try:
        if tipo == "VIEW":
            raise ValueError("vista: se documenta columna por columna")
        cambios = ", ".join(f"`{c}` COMMENT '{sql_texto(t)}'" for c, t in cols)
        spark.sql(f"ALTER TABLE {nombre} ALTER COLUMN {cambios}")
        documentadas += len(cols)
    except Exception:
        for c, t in cols:
            try:
                spark.sql(f"COMMENT ON COLUMN {nombre}.`{c}` IS '{sql_texto(t)}'")
                documentadas += 1
            except Exception as e:
                errores.append((f"{esquema}.{tabla}.{c}", str(e).splitlines()[0][:150]))

print(f"Columnas documentadas: {documentadas} de {len(columnas)}")
print(f"Sin descripción en los diccionarios: {len(sin_descripcion)}", sin_descripcion[:20])
print(f"Errores al aplicar: {len(errores)}")
for col, err in errores[:10]:
    print("  ", col, "->", err)

## 5. Diccionario de datos
Esta es la vista que pide el entregable: **tabla, columna, tipo de dato y descripción**, leída directamente de Unity Catalog (no de un Excel que alguien tenga que mantener).

In [0]:
diccionario = spark.sql(f"""
    SELECT c.table_schema AS capa,
           c.table_name   AS tabla,
           t.comment      AS descripcion_tabla,
           c.column_name  AS columna,
           c.full_data_type AS tipo,
           c.is_nullable  AS admite_nulos,
           c.comment      AS descripcion
    FROM {catalogo}.information_schema.columns c
    JOIN {catalogo}.information_schema.tables t
      ON t.table_schema = c.table_schema AND t.table_name = c.table_name
    WHERE c.table_schema IN ('silver', 'gold')
      AND c.table_name NOT LIKE '%_cuarentena'
    ORDER BY CASE c.table_schema WHEN 'silver' THEN 1 ELSE 2 END, c.table_name, c.ordinal_position""")

display(diccionario)

### 5.1 · Diccionario de Bronze
Mismo formato para la capa cruda: aquí se ve que **todo llega como `string`**, que es justamente lo que Silver resuelve.

In [0]:
display(spark.sql(f"""
    SELECT c.table_name AS tabla, c.column_name AS columna,
           c.full_data_type AS tipo, c.comment AS descripcion
    FROM {catalogo}.information_schema.columns c
    WHERE c.table_schema = 'bronze'
    ORDER BY c.table_name, c.ordinal_position"""))

## 6. Cobertura de la documentación
Porcentaje de columnas con descripción por capa. El objetivo es **100 %**.

In [0]:
cobertura = spark.sql(f"""
    SELECT c.table_schema AS capa,
           count(DISTINCT c.table_name) AS tablas,
           count(*) AS columnas,
           count(c.comment) AS columnas_documentadas,
           round(100 * count(c.comment) / count(*), 1) AS pct_documentado
    FROM {catalogo}.information_schema.columns c
    WHERE c.table_schema IN ('bronze', 'silver', 'gold')
    GROUP BY c.table_schema
    ORDER BY CASE c.table_schema WHEN 'bronze' THEN 1 WHEN 'silver' THEN 2 ELSE 3 END""")
display(cobertura)

tablas_sin_desc = spark.sql(f"""
    SELECT table_schema, table_name FROM {catalogo}.information_schema.tables
    WHERE table_schema IN ('bronze', 'silver', 'gold') AND comment IS NULL""").collect()
print("Tablas sin descripción:", [f"{r.table_schema}.{r.table_name}" for r in tablas_sin_desc] or "ninguna")

In [0]:
display(spark.sql(f"""
    SELECT schema_name AS capa, table_name AS tabla, column_name AS columna, tag_name, tag_value
    FROM {catalogo}.information_schema.column_tags
    ORDER BY capa, tabla, columna"""))